In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import datetime
import scipy.stats
#import plotly.offline as pyo
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as cl
import matplotlib.dates as mdates
import matplotlib
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import seaborn as sns
import netCDF4 as nc
import xarray as xr
#import cartopy.crs as ccrs
#from cartopy import feature as cfa
import os
import plotly.io as pio
pio.renderers.default = 'notebook'

In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    decimal_part = decimal_year - year
    start_of_year = pd.to_datetime([f'{y}-01-01' for y in year])
    seconds_in_year = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - start_of_year).total_seconds()
    seconds = decimal_part * seconds_in_year
    return start_of_year + pd.to_timedelta(seconds, unit='s')

def datetime_to_decimal_year(datetime_):
    year = datetime_.year
    days = (datetime_.date() - datetime.date(year, 1,1)).days
    res = year + days/365.25
    return res


def FFT_res(x, si):
    n = len(x)
    x_FFT_shifted = np.fft.fftshift(np.fft.fft(x))
    x_FFT_freq_shifted = np.fft.fftshift(np.fft.fftfreq(n, d=si))
    return x_FFT_freq_shifted, x_FFT_shifted


def fft_noramlize(fft_res):
    n = len(fft_res)
    norm_fft = np.abs(fft_res)/(np.max(np.abs(fft_res)))
    return norm_fft

def line_fit(X,Y):
    P = np.polyfit(X,Y,1)
    dx = np.max(X) - np.min(X)
    X_new = np.array([np.min(X)-dx*0.05, np.max(X)+dx*0.05])
    Y_new = np.polyval(P, X_new)
    return X_new, Y_new, P

In [ ]:
df_updated = pd.read_excel('../../2025-12-20_AgeModel-NewXrayPatch/age_model/20260218/interp_20260218.xlsx')
df_sorted_updated = df_updated.sort_values('age_absolute')
df_sorted_updated

In [ ]:
%run ../../2025-12-20_AgeModel-NewXrayPatch/age_model/20260215_read_data.py

In [ ]:
#MODEL_first = pd.read_excel('../../2026-02-03_Age-Model/claude/coral_age_warped.xlsx')
MODEL = pd.read_excel('../../2026-02-03_Age-Model/Python/claude/coral_age_warped.xlsx')
MODEL = MODEL.sort_values('age_warped')
MODEL

In [ ]:
MODEL['age_warped_dt'] = decimal_year_to_datetime(MODEL['age_warped'])
MODEL['d18o_ma'] = MODEL['d18o'].rolling(3, center=True).mean()
MODEL['d13c_ma'] = MODEL['d13c'].rolling(3, center=True).mean()
MODEL['age_warped_ma'] = MODEL['age_warped'].rolling(3, center=True).mean()
MODEL['age_wraped_month'] = MODEL['age_warped_dt'].dt.month
MODEL_monthly= MODEL[['d13c', 'd18o', 'age_wraped_month']].groupby('age_wraped_month').mean()
MODEL_monthly

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), subplot_kw={'projection': 'polar'})

ax = axes[0]

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

theta = np.linspace(0, 2*np.pi, 12, endpoint=False)
width = 2*np.pi/12 * 0.9  # 90% of bin width (small gap)

rmin, rmax = 25.45, 25.8
vals = MODEL_monthly['d18o'].values

# Make bars start at rmin, and only show the variation within [rmin, rmax]
heights = np.clip(vals, rmin, rmax) - rmin

ax.bar(theta, heights, width=width, bottom=rmin,
       color='steelblue', alpha=0.7, edgecolor='black', linewidth=1.5)

ax.set_rlim(rmin, rmax)

ax.set_rticks([])
ax.set_xticks(theta)
ax.set_xticklabels(months)

ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)

ax.set_title(r'$\delta^{18}$O')


ax = axes[1]


rmin, rmax = -1.45, -1.18
vals = MODEL_monthly['d13c'].values

# Make bars start at rmin, and only show the variation within [rmin, rmax]
heights = np.clip(vals, rmin, rmax) - rmin

ax.bar(theta, heights, width=width, bottom=rmin,
       color='green', alpha=0.7, edgecolor='black', linewidth=1.5)

ax.set_rlim(rmin, rmax)

ax.set_rticks([])
ax.set_xticks(theta)
ax.set_xticklabels(months)

ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)

ax.set_title(r'$\delta^{13}$C')

plt.suptitle('Bayesian Age Model', y= 1.05 , fontsize=16)


#fig.savefig('claude_age_model_d13c_d18o_monthly_means.pdf',bbox_inches='tight' )

In [ ]:
climate = pd.read_csv('../../2026-01-15_Climate-Model/analysis/climate_df.csv')
climate['datetime'] = pd.to_datetime(climate['datetime'])
climate['datetime_month'] = climate['datetime'].dt.month
climate_monthly = climate.groupby('datetime_month').mean()
climate_monthly

In [ ]:
MODEL['age_wraped_month'].value_counts()

In [ ]:
fig, ax = plt.subplots(1, figsize=(6,4))


ax.plot(climate_monthly['synthetic_d18o'], color='blue',label='climate')
ax.plot(MODEL_monthly['d18o']-MODEL_monthly['d18o'].mean(), '--', color='blue', label='model')
ax.legend(loc='lower right')


ax2 = ax.twinx()
#ax2.plot(climate_monthly['d13c'], color='green')
ax2.plot(MODEL_monthly['d13c'],  '--',color='green',label='model')


In [ ]:
a_1 = -0.22
#a_2 = 0.45
a_2 = 0.8

synth_d18o = climate['synthetic_d18o']



fig = make_subplots(specs=[[{"secondary_y": True}]])

#fig.add_trace(go.Scatter(x=df_sorted['age_dt'], y=df_sorted['d18o']-df_sorted['d18o'].mean(),  mode="markers", marker_color="#3492eb",  name='d18O', text=df_sorted['Name']),secondary_y=False)
#fig.add_trace(go.Scatter(x=df_resampled.index.values, y=df_resampled['d18o_ma']-df_resampled['d18o_ma'].mean(), mode="lines", line_color='#134b80', name='average'), secondary_y=False)
#fig.add_trace(go.Scatter(x=df_resampled.index.values, y=df_resampled['d18o_ma2']-df_resampled['d18o_ma2'].mean(), mode="lines", line_color='navy', name='average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=df_sorted['age_dt'], y=df_sorted['d18o_no_tectonic'] ,  mode="markers", marker_color="#3492eb",  name='d18O', text=df_sorted['Name']),secondary_y=False)
#fig.add_trace(go.Scatter(x=df_sorted['age_dt'], y=df_sorted['d18o_notec_ma'] , mode="lines", line_color='#134b80', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=MODEL['age_warped'], y=MODEL['d18o'] , mode="markers", line_color='#3492eb',text=MODEL['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=MODEL['age_warped_ma'], y=MODEL['d18o_ma'] , mode="lines", line_color='#134b80', name='average'), secondary_y=False)



fig.add_trace(go.Scatter(x=climate['age_absolute'], y= climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=2000, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O-CONSTANT-TEMP.html')


In [ ]:
MODEL.to_csv('age_model_edited.csv')

In [ ]:
fig, ax = plt.subplots(1, figsize=(12, 5))

ax.plot(df_sorted['age_absolute'], df_sorted['d18o'],'.', label='old')
ax.plot(df_sorted_updated['age_absolute'], df_sorted_updated['d18o'],'.', label='new')
ax.legend()

ax.set_xlim(2000, 2007)